# Statistical Inference, Calibration, and Model Checking
## From an estimated effect to a defensible model claim

### Learning goals

You will distinguish estimands from estimators, inspect sampling distributions and interval coverage, connect sample size to power, read likelihood curvature, measure and repair calibration, and use predictive checks to discover a misspecified independence model.

Complete Notebook 20 first. Predict the direction of every result before running its simulation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.figsize': (7, 4.5), 'axes.grid': True})
rng = np.random.default_rng(7)
from statistics import NormalDist

## 1. Estimands, estimators, and coverage

The estimand below is the population mean paired improvement. The sample mean difference is an estimator; its realized value is an estimate. Repeated simulation reveals its sampling distribution and whether a confidence procedure achieves its advertised long-run coverage.

In [ ]:
true_effect=0.35; pair_sd=1.4; n=64; repetitions=5000
samples=rng.normal(true_effect,pair_sd,size=(repetitions,n))
estimates=samples.mean(axis=1)
ses=samples.std(axis=1,ddof=1)/np.sqrt(n)
lower=estimates-1.96*ses; upper=estimates+1.96*ses
coverage=np.mean((lower<=true_effect)&(true_effect<=upper))
print('mean estimate:',estimates.mean(),'empirical SE:',estimates.std(ddof=1),'coverage:',coverage)
plt.figure()
plt.hist(estimates,bins=45,density=True,alpha=.75)
plt.axvline(true_effect,color='C3',label='estimand');plt.legend();plt.title('Sampling distribution of paired mean');plt.show()
assert abs(estimates.mean()-true_effect)<0.02
assert abs(estimates.std(ddof=1)-pair_sd/np.sqrt(n))<0.01
assert 0.94<coverage<0.96

## 2. Power is a property of a procedure under an alternative

Power depends on effect size, noise, design, sample size, and the rejection rule. It is not something a nonsignificant study can recover after the fact. The simulation uses a two-sided z threshold to isolate the sample-size relationship.

In [ ]:
def simulated_power(effect,n,sd=1.4,reps=12000):
    means=rng.normal(effect,sd/np.sqrt(n),size=reps)
    z=means/(sd/np.sqrt(n))
    return np.mean(np.abs(z)>1.96)

sizes=np.array([16,32,64,128,256])
powers=np.array([simulated_power(true_effect,int(size)) for size in sizes])
plt.figure()
plt.plot(sizes,powers,'o-');plt.axhline(.8,color='C3',ls='--');plt.ylim(0,1)
plt.xlabel('paired sample size');plt.ylabel('power');plt.title('Power for effect 0.35');plt.show()
print(dict(zip(sizes,powers.round(3))))
assert np.all(np.diff(powers)>0)
assert powers[-1]>.95
assert abs(simulated_power(0,64)-.05)<.015

## 3. Likelihood geometry and information

For $k$ successes in $n$ Bernoulli trials, $\ell(p)=k\log p+(n-k)\log(1-p)$. Its maximizer is $k/n$; local curvature measures how sharply nearby parameters are distinguished. More data make the likelihood narrower.

In [ ]:
def bernoulli_loglik(p,k,n): return k*np.log(p)+(n-k)*np.log1p(-p)
grid=np.linspace(.01,.99,500)
fig,ax=plt.subplots()
for n,k in [(20,14),(200,140)]:
    ll=bernoulli_loglik(grid,k,n); ll-=ll.max()
    ax.plot(grid,ll,label=f'n={n}')
ax.axvline(.7,color='black',ls=':');ax.set_ylim(-25,1);ax.set_ylabel('relative log likelihood');ax.legend();plt.show()

for n,k in [(20,14),(200,140)]:
    p_hat=k/n
    observed_information=k/p_hat**2+(n-k)/(1-p_hat)**2
    print(n,'MLE',p_hat,'local SE approximation',1/np.sqrt(observed_information))
assert np.isclose(14/20,0.7)
assert np.isclose((140/.7**2+60/.3**2)/(14/.7**2+6/.3**2),10)

## 4. Calibration is different from discrimination

A calibrated 70% prediction should be correct about 70% of the time among comparable cases. Reliability diagrams compare predicted probabilities with empirical frequencies. Platt-style logistic calibration estimates an intercept and slope on held-out calibration data; using the evaluation set would leak information.

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))
def logit(p):
    p=np.clip(p,1e-6,1-1e-6); return np.log(p/(1-p))
def reliability(probs,outcomes,bins=10):
    edges=np.linspace(0,1,bins+1); ids=np.clip(np.digitize(probs,edges)-1,0,bins-1)
    pred=[]; obs=[]; weight=[]
    for j in range(bins):
        mask=ids==j
        if mask.any(): pred.append(probs[mask].mean());obs.append(outcomes[mask].mean());weight.append(mask.mean())
    return np.array(pred),np.array(obs),np.array(weight)
def ece(probs,outcomes):
    p,o,w=reliability(probs,outcomes); return np.sum(w*np.abs(p-o))
def fit_calibrator(probs,outcomes,steps=20):
    Z=np.column_stack([np.ones(len(probs)),logit(probs)]); beta=np.array([0.,1.])
    for _ in range(steps):
        q=sigmoid(Z@beta); grad=Z.T@(q-outcomes); H=Z.T@(Z*(q*(1-q))[:,None])+1e-6*np.eye(2)
        beta-=np.linalg.solve(H,grad)
    return beta

raw=rng.beta(2,2,size=6000)
true_prob=sigmoid(-0.5+0.65*logit(raw))
outcome=rng.binomial(1,true_prob)
cal=slice(0,3000); test=slice(3000,None)
beta=fit_calibrator(raw[cal],outcome[cal])
adjusted=sigmoid(beta[0]+beta[1]*logit(raw[test]))
before=ece(raw[test],outcome[test]); after=ece(adjusted,outcome[test])
p0,o0,_=reliability(raw[test],outcome[test]);p1,o1,_=reliability(adjusted,outcome[test])
plt.figure()
plt.plot([0,1],[0,1],'k--');plt.plot(p0,o0,'o-',label='raw');plt.plot(p1,o1,'o-',label='calibrated')
plt.xlabel('mean prediction');plt.ylabel('observed frequency');plt.legend();plt.show()
print('fitted intercept/slope:',beta,'ECE before/after:',before,after)
assert before>0.05 and after<before/2
assert abs(beta[0]+0.5)<0.12 and abs(beta[1]-.65)<0.12

## 5. Predictive checks can reject a model that fits its aggregate count

An iid Bernoulli model treats order as irrelevant. The observed sequence below has exactly 50% successes, so its count fits $p=0.5$ perfectly, but its clustering is suspicious. A posterior predictive check asks whether replicated data from the fitted model reproduce the longest run.

In [ ]:
observed=np.tile(np.r_[np.ones(25,dtype=int),np.zeros(25,dtype=int)],2)
successes=observed.sum(); total=len(observed)
alpha,beta_prior=1,1
post_a,post_b=alpha+successes,beta_prior+total-successes

def longest_run(row):
    best=cur=1
    for j in range(1,len(row)):
        cur=cur+1 if row[j]==row[j-1] else 1; best=max(best,cur)
    return best

replicates=2500
p_rep=rng.beta(post_a,post_b,size=replicates)
rep=rng.binomial(1,p_rep[:,None],size=(replicates,total))
rep_runs=np.array([longest_run(row) for row in rep])
observed_run=longest_run(observed)
ppp=np.mean(rep_runs>=observed_run)
plt.figure()
plt.hist(rep_runs,bins=np.arange(rep_runs.min(),rep_runs.max()+2)-.5)
plt.axvline(observed_run,color='C3',label=f'observed run={observed_run}');plt.legend();plt.show()
print('posterior:',post_a,post_b,'predictive tail probability:',ppp)
assert successes/total==0.5
assert observed_run==25
assert ppp<0.01

## Cumulative evidence audit

Suppose Model B has a positive paired mean improvement, a confidence interval barely excluding zero, higher power in a planned replication, improved calibration after held-out recalibration, and a failed run-length predictive check on one deployment slice.

Before writing “B is better,” separate:

1. **estimand:** better on which population, outcome, and decision threshold?
2. **estimation:** effect size and uncertainty, not only a p-value
3. **design:** pairing, selection, multiplicity, and planned sample size
4. **prediction quality:** discrimination versus calibration
5. **model adequacy:** which dependence or distributional assumption failed
6. **action:** deploy, gather evidence, restrict scope, or reject

In [ ]:
# A compact reusable audit record. Replace the values with a real evaluation.
audit={
    'estimand':'mean paired score difference on the target prompt population',
    'estimate':0.35,
    'interval':(0.04,0.66),
    'planned_replication_n':128,
    'estimated_power':float(powers[sizes==128][0]),
    'held_out_calibration_ece':float(after),
    'model_check_tail_probability':float(ppp),
    'decision':'do not make an unconditional superiority claim; investigate dependence and slice shift',
}
for key,value in audit.items(): print(f'{key}: {value}')
assert audit['interval'][0]>0
assert audit['estimated_power']>.7
assert audit['model_check_tail_probability']<.01
assert 'unconditional' in audit['decision']

### Cumulative explanation prompts

- Why can a confidence procedure have 95% coverage while a particular interval is narrow or misleading under violated assumptions?
- Which changes increase power without changing the true effect?
- Why does a sharper likelihood not guarantee a better-specified model?
- Why must calibration be fitted and evaluated on different data?
- What did the run-length check detect that the success count could not?

Give a two-minute evidence briefing that states the claim, uncertainty, assumptions, failed check, and next decision.